# CNN-EXP-037 — Frozen WIDTH64 Hard-FP Reranker

## Гипотеза

Проверяем только **objective + hard-negative sampling**.

```text
Frozen WIDTH64 backbone
RF = 1029
        ↓
64-D feature / nucleotide
        ↓
Linear presence head
64 → 1
```

Не добавляем MLP, не меняем backbone, не переходим на WIDTH128.

### Heads

```text
A. Weighted BCE
   all TP + random TN + train HARD_FP

B. Weighted BCE
   + λ · softplus(-(score_TP - score_HARD_FP))
```

Grid:

```text
hard fraction = 25%, 50%
lambda = 0, 0.1, 0.3, 1.0
```

`lambda=0` — BCE-only.

LOW_TP входит в обычные positives, но отдельной loss/oversampling для LOW_TP нет.
LOW_TP здесь только diagnostic group.

Конфигурация выбирается **по AP на естественном фоне held-out train-contig**.
Validation используется один раз после выбора.


In [1]:
from pathlib import Path
import copy
import gc
import json
import math
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

import torch
from torch import nn
import torch.nn.functional as F
import torch_directml

from scipy.stats import pearsonr, rankdata

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
)
from ml_project.epic_data import load_split

device = torch_directml.device()

print("Project:", PROJECT_ROOT)
print("DirectML:", device)
print("GPU:", torch_directml.device_name(0))


Project: D:\Projects\EPIC\EPIC
DirectML: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [2]:
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

EXP036_RUN = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP036_WIDTH64_RF1029"
    / "run_001"
)

EXP036_CHECKPOINT = EXP036_RUN / "width64_final.pt"
EXP036_RESULT = EXP036_RUN / "EXP036_WIDTH64_RESULT.json"

assert EXP036_CHECKPOINT.is_file(), EXP036_CHECKPOINT

checkpoint = torch.load(
    EXP036_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

frozen_model = create_cnn_presence_intensity()
frozen_model.load_state_dict(checkpoint["model_state_dict"])

assert frozen_model.backbone.stem.out_channels == 64
assert frozen_model.profile_head.in_channels == 64
assert frozen_model.profile_head.out_channels == 2

rf = 1
for block in frozen_model.backbone.blocks:
    for conv in (block.conv1, block.conv2):
        rf += (int(conv.kernel_size[0]) - 1) * int(conv.dilation[0])

assert rf == 1029

for p in frozen_model.parameters():
    p.requires_grad_(False)

frozen_model = frozen_model.to(device)
frozen_model.eval()

old_presence_weight = (
    frozen_model.profile_head.weight[0, :, 0]
    .detach().cpu().numpy().astype(np.float32, copy=True)
)
old_presence_bias = float(
    frozen_model.profile_head.bias[0].detach().cpu().item()
)

print("Checkpoint:", EXP036_CHECKPOINT)
print("Channels:", frozen_model.backbone.stem.out_channels)
print("RF:", rf)
print("Backbone frozen:", all(not p.requires_grad for p in frozen_model.backbone.parameters()))

if EXP036_RESULT.is_file():
    exp036_result = json.loads(EXP036_RESULT.read_text(encoding="utf-8"))
    display(pd.DataFrame([exp036_result["overall"]]))


Checkpoint: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP036_WIDTH64_RF1029\run_001\width64_final.pt
Channels: 64
RF: 1029
Backbone frozen: True


,positive_n,presence_AP_5dp,presence_EPIC_Spearman_raw,presence_EPIC_Spearman_5dp,intensity_EPIC_Spearman_raw
0,56205,0.033997,0.177041,0.152906,0.188709


In [3]:
train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

train_tiles = train_generator.tile_index.copy().reset_index(drop=True)
validation_tiles = validation_generator.tile_index.copy().reset_index(drop=True)

TRAIN_CONTIGS = sorted(train_tiles["contig"].astype(str).unique())
VALIDATION_CONTIGS = sorted(validation_tiles["contig"].astype(str).unique())

print("Train contigs:", len(TRAIN_CONTIGS), TRAIN_CONTIGS)
print("Validation contigs:", len(VALIDATION_CONTIGS), VALIDATION_CONTIGS)
print("Train tiles:", len(train_tiles))
print("Validation tiles:", len(validation_tiles))


Train contigs: 9 ['NC_064035.1', 'NC_064036.1', 'NC_064038.1', 'NC_064039.1', 'NC_064040.1', 'NC_064043.1', 'NC_064045.1', 'NC_064047.1', 'NC_064048.1']
Validation contigs: 3 ['NC_064034.1', 'NC_064041.1', 'NC_064042.1']
Train tiles: 130525
Validation tiles: 47743


## 1. Train feature bank

По exhaustive train tiles собираем:

- **все TP**;
- случайные TN;
- HARD_FP candidates по старой WIDTH64 presence-head.

HARD_FP берутся только из train.


In [4]:
EXPERIMENT_ID = "CNN-EXP-037"
EXPERIMENT_SLUG = "CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER"

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / EXPERIMENT_SLUG
    / "run_001"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_BANK_PATH = RUN_DIR / "train_feature_bank_width64.npz"

RANDOM_TN_PER_TILE_STRAND = 4
HARD_CANDIDATES_PER_TILE_STRAND = 4
MAX_HARD_FP_PER_CONTIG = 50_000
FEATURE_PROGRESS_EVERY = 250

print("Artifacts:", RUN_DIR)
print("Feature bank:", FEATURE_BANK_PATH)


Artifacts: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER\run_001
Feature bank: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER\run_001\train_feature_bank_width64.npz


In [5]:
def sigmoid_numpy(x):
    x = np.asarray(x, dtype=np.float64)
    out = np.empty_like(x, dtype=np.float64)

    positive = x >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))

    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)

    return out.astype(np.float32)


def extract_backbone_and_old_outputs(model, batch):
    # Returns central-crop CPU arrays:
    # features       [2B, L, 64]
    # presence_logit [2B, L]
    # intensity      [2B, L]
    x_gpu = batch.x_both_strands.to(device)

    feature_gpu = model.backbone(x_gpu)
    output_gpu = model.profile_head(feature_gpu)

    c = profile_config.context_flank
    L = profile_config.target_length

    feature = (
        feature_gpu[:, :, c:c + L]
        .permute(0, 2, 1)
        .detach().cpu().numpy()
        .astype(np.float32, copy=True)
    )

    output = (
        output_gpu[:, :, c:c + L]
        .detach().cpu().numpy()
        .astype(np.float32, copy=True)
    )

    presence_logit = output[:, 0, :]
    intensity = output[:, 1, :]

    del x_gpu, feature_gpu, output_gpu

    return feature, presence_logit, intensity


In [6]:
if FEATURE_BANK_PATH.is_file():
    print("Feature bank already exists:", FEATURE_BANK_PATH)

else:
    rng = np.random.default_rng(20260924)

    pos_X = []
    pos_old_score = []
    pos_count = []
    pos_contig = []

    random_X = []
    random_old_score = []
    random_contig = []

    hard_X = []
    hard_old_score = []
    hard_contig = []

    started = time.perf_counter()
    positions_seen = 0

    frozen_model.eval()

    with torch.no_grad():
        for row_i in range(len(train_tiles)):
            selected = train_tiles.iloc[row_i:row_i + 1]
            batch = train_generator.make_batch(selected)

            features, old_logit, _ = extract_backbone_and_old_outputs(
                frozen_model,
                batch,
            )

            old_probability = sigmoid_numpy(old_logit)

            plus_count = (
                batch.count[0, 0, :]
                .numpy().astype(np.float32, copy=False)
            )
            minus_count_reverse = (
                batch.count[0, 1, :]
                .flip(-1)
                .numpy().astype(np.float32, copy=False)
            )

            plus_mask = (
                batch.mask[0, 0, :]
                .numpy().astype(bool, copy=False)
            )
            minus_mask_reverse = (
                batch.mask[0, 1, :]
                .flip(-1)
                .numpy().astype(bool, copy=False)
            )

            contig = str(selected.iloc[0]["contig"])

            for orient_i, (count, mask) in enumerate(
                (
                    (plus_count, plus_mask),
                    (minus_count_reverse, minus_mask_reverse),
                )
            ):
                feat = features[orient_i]
                probability = old_probability[orient_i]

                valid_index = np.flatnonzero(mask)

                positive_index = valid_index[
                    count[valid_index] > 0
                ]
                negative_index = valid_index[
                    count[valid_index] == 0
                ]

                if len(positive_index):
                    pos_X.append(feat[positive_index].copy())
                    pos_old_score.append(probability[positive_index].copy())
                    pos_count.append(count[positive_index].copy())
                    pos_contig.extend([contig] * len(positive_index))

                if len(negative_index):
                    n_random = min(
                        RANDOM_TN_PER_TILE_STRAND,
                        len(negative_index),
                    )
                    random_index = rng.choice(
                        negative_index,
                        size=n_random,
                        replace=False,
                    )

                    random_X.append(feat[random_index].copy())
                    random_old_score.append(probability[random_index].copy())
                    random_contig.extend([contig] * len(random_index))

                    n_hard = min(
                        HARD_CANDIDATES_PER_TILE_STRAND,
                        len(negative_index),
                    )
                    negative_score = probability[negative_index]

                    if len(negative_index) <= n_hard:
                        hard_index = negative_index
                    else:
                        local_top = np.argpartition(
                            negative_score,
                            -n_hard,
                        )[-n_hard:]
                        hard_index = negative_index[local_top]

                    hard_X.append(feat[hard_index].copy())
                    hard_old_score.append(probability[hard_index].copy())
                    hard_contig.extend([contig] * len(hard_index))

                positions_seen += int(mask.sum())

            del (
                batch,
                selected,
                features,
                old_logit,
                old_probability,
                plus_count,
                minus_count_reverse,
                plus_mask,
                minus_mask_reverse,
            )

            done = row_i + 1

            if (
                done == 1
                or done % FEATURE_PROGRESS_EVERY == 0
                or done == len(train_tiles)
            ):
                elapsed = time.perf_counter() - started
                print(
                    f"tiles={done:>6}/{len(train_tiles)} "
                    f"positions={positions_seen:>13,} "
                    f"positions/s={positions_seen / elapsed:,.0f}"
                )

            if done % 50 == 0:
                gc.collect()

    pos_X = np.concatenate(pos_X, axis=0)
    pos_old_score = np.concatenate(pos_old_score)
    pos_count = np.concatenate(pos_count)
    pos_contig = np.asarray(pos_contig)

    random_X = np.concatenate(random_X, axis=0)
    random_old_score = np.concatenate(random_old_score)
    random_contig = np.asarray(random_contig)

    hard_X = np.concatenate(hard_X, axis=0)
    hard_old_score = np.concatenate(hard_old_score)
    hard_contig = np.asarray(hard_contig)

    hard_keep_parts = []

    for contig in TRAIN_CONTIGS:
        idx = np.flatnonzero(hard_contig == contig)

        if len(idx) > MAX_HARD_FP_PER_CONTIG:
            local = np.argpartition(
                hard_old_score[idx],
                -MAX_HARD_FP_PER_CONTIG,
            )[-MAX_HARD_FP_PER_CONTIG:]
            idx = idx[local]

        hard_keep_parts.append(idx)

    hard_keep = np.concatenate(hard_keep_parts)

    hard_X = hard_X[hard_keep]
    hard_old_score = hard_old_score[hard_keep]
    hard_contig = hard_contig[hard_keep]

    np.savez_compressed(
        FEATURE_BANK_PATH,
        pos_X=pos_X,
        pos_old_score=pos_old_score,
        pos_count=pos_count,
        pos_contig=pos_contig,
        random_X=random_X,
        random_old_score=random_old_score,
        random_contig=random_contig,
        hard_X=hard_X,
        hard_old_score=hard_old_score,
        hard_contig=hard_contig,
    )

    print("TP:", pos_X.shape)
    print("Random TN:", random_X.shape)
    print("HARD_FP:", hard_X.shape)
    print("Saved:", FEATURE_BANK_PATH)


tiles=     1/130525 positions=          498 positions/s=12,776
tiles=   250/130525 positions=      374,670 positions/s=236,285
tiles=   500/130525 positions=      811,248 positions/s=253,131
tiles=   750/130525 positions=    1,209,988 positions/s=258,785
tiles=  1000/130525 positions=    1,606,742 positions/s=261,324
tiles=  1250/130525 positions=    2,025,652 positions/s=268,006
tiles=  1500/130525 positions=    2,447,804 positions/s=273,437
tiles=  1750/130525 positions=    2,875,410 positions/s=279,021
tiles=  2000/130525 positions=    3,276,578 positions/s=279,260
tiles=  2250/130525 positions=    3,727,040 positions/s=284,197
tiles=  2500/130525 positions=    4,184,270 positions/s=289,039
tiles=  2750/130525 positions=    4,626,284 positions/s=291,881
tiles=  3000/130525 positions=    5,062,232 positions/s=294,046
tiles=  3250/130525 positions=    5,524,502 positions/s=297,525
tiles=  3500/130525 positions=    5,960,950 positions/s=298,534
tiles=  3750/130525 positions=    6,421,7

In [7]:
bank = np.load(FEATURE_BANK_PATH, allow_pickle=False)

pos_X = bank["pos_X"]
pos_old_score = bank["pos_old_score"]
pos_count = bank["pos_count"]
pos_contig = bank["pos_contig"]

random_X = bank["random_X"]
random_old_score = bank["random_old_score"]
random_contig = bank["random_contig"]

hard_X = bank["hard_X"]
hard_old_score = bank["hard_old_score"]
hard_contig = bank["hard_contig"]

print("TP:", pos_X.shape)
print("Random TN:", random_X.shape)
print("HARD_FP:", hard_X.shape)

display(
    pd.DataFrame(
        {
            "contig": TRAIN_CONTIGS,
            "TP": [int(np.sum(pos_contig == c)) for c in TRAIN_CONTIGS],
            "random_TN": [int(np.sum(random_contig == c)) for c in TRAIN_CONTIGS],
            "HARD_FP": [int(np.sum(hard_contig == c)) for c in TRAIN_CONTIGS],
        }
    )
)


TP: (170443, 64)
Random TN: (1044002, 64)
HARD_FP: (450000, 64)


,contig,TP,random_TN,HARD_FP
0,NC_064035.1,30987,151318,50000
1,NC_064036.1,26323,142686,50000
2,NC_064038.1,14793,117192,50000
3,NC_064039.1,21975,123984,50000
4,NC_064040.1,19067,118478,50000
5,NC_064043.1,19558,109086,50000
6,NC_064045.1,15184,95862,50000
7,NC_064047.1,11668,101970,50000
8,NC_064048.1,10888,83426,50000


## 2. Linear head training

Стандартизация остаётся линейным преобразованием и затем сворачивается обратно
в один `Conv1d(64,1,1)`.

Инициализация головы эквивалентна старой WIDTH64 presence-head.


In [8]:
HEAD_SEED = 20260924
NEGATIVE_MULTIPLIER = 4.0
HEAD_EPOCHS = 40
HEAD_BATCH_SIZE = 8192
HEAD_LR = 3e-3
WEIGHT_DECAY = 1e-4

HARD_FRACTIONS = (0.25, 0.50)
LAMBDAS = (0.0, 0.1, 0.3, 1.0)

CONFIGS = [
    {
        "name": (
            f"hard{int(hard_fraction * 100):02d}_"
            f"lambda{str(lam).replace('.', 'p')}"
        ),
        "hard_fraction": hard_fraction,
        "lambda_rank": lam,
    }
    for hard_fraction in HARD_FRACTIONS
    for lam in LAMBDAS
]

display(pd.DataFrame(CONFIGS))


def fit_scaler(X):
    mean = X.mean(axis=0, dtype=np.float64).astype(np.float32)
    std = X.std(axis=0, dtype=np.float64).astype(np.float32)
    std = np.maximum(std, 1e-6)
    return mean, std


def old_head_in_standardized_space(mean, std):
    w = (old_presence_weight * std).astype(np.float32)
    b = np.float32(
        old_presence_bias
        + np.dot(old_presence_weight, mean)
    )
    return w, b


def train_linear_head(
    *,
    train_pos_X,
    train_random_X,
    train_hard_X,
    hard_fraction,
    lambda_rank,
    seed,
):
    rng = np.random.default_rng(seed)

    scaler_source = np.concatenate(
        (train_pos_X, train_random_X, train_hard_X),
        axis=0,
    )
    mean, std = fit_scaler(scaler_source)
    del scaler_source

    n_pos = len(train_pos_X)
    n_neg_target = int(round(NEGATIVE_MULTIPLIER * n_pos))

    n_hard_target = int(round(hard_fraction * n_neg_target))
    n_random_target = n_neg_target - n_hard_target

    hard_index = rng.choice(
        len(train_hard_X),
        size=n_hard_target,
        replace=n_hard_target > len(train_hard_X),
    )
    random_index = rng.choice(
        len(train_random_X),
        size=n_random_target,
        replace=n_random_target > len(train_random_X),
    )

    X_neg = np.concatenate(
        (
            train_hard_X[hard_index],
            train_random_X[random_index],
        ),
        axis=0,
    )

    X_pos = train_pos_X

    X = np.concatenate((X_pos, X_neg), axis=0)
    y = np.concatenate(
        (
            np.ones(len(X_pos), dtype=np.float32),
            np.zeros(len(X_neg), dtype=np.float32),
        )
    )

    X = ((X - mean) / std).astype(np.float32, copy=False)

    X_hard_pair = (
        (train_hard_X - mean) / std
    ).astype(np.float32, copy=False)

    X_pos_pair = (
        (train_pos_X - mean) / std
    ).astype(np.float32, copy=False)

    linear = nn.Linear(64, 1, bias=True)

    init_w, init_b = old_head_in_standardized_space(mean, std)

    with torch.no_grad():
        linear.weight.copy_(torch.from_numpy(init_w)[None, :])
        linear.bias.fill_(float(init_b))

    optimizer = torch.optim.AdamW(
        linear.parameters(),
        lr=HEAD_LR,
        weight_decay=WEIGHT_DECAY,
    )

    pos_weight = torch.tensor(
        [len(X_neg) / max(len(X_pos), 1)],
        dtype=torch.float32,
    )

    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    torch.manual_seed(seed)
    n = len(X)

    for epoch in range(HEAD_EPOCHS):
        order = rng.permutation(n)

        for start in range(0, n, HEAD_BATCH_SIZE):
            idx = order[start:start + HEAD_BATCH_SIZE]

            xb = torch.from_numpy(X[idx])
            yb = torch.from_numpy(y[idx])[:, None]

            optimizer.zero_grad(set_to_none=True)

            logit = linear(xb)
            loss = bce(logit, yb)

            if lambda_rank > 0 and len(X_hard_pair) > 0:
                pair_n = min(
                    HEAD_BATCH_SIZE,
                    len(X_pos_pair),
                    len(X_hard_pair),
                )

                pos_idx = rng.choice(
                    len(X_pos_pair),
                    size=pair_n,
                    replace=False,
                )
                hard_idx = rng.choice(
                    len(X_hard_pair),
                    size=pair_n,
                    replace=False,
                )

                pos_logit = linear(
                    torch.from_numpy(X_pos_pair[pos_idx])
                )
                hard_logit = linear(
                    torch.from_numpy(X_hard_pair[hard_idx])
                )

                rank_loss = F.softplus(
                    -(pos_logit - hard_logit)
                ).mean()

                loss = loss + lambda_rank * rank_loss

            loss.backward()
            optimizer.step()

    w_std = (
        linear.weight[0]
        .detach().cpu().numpy()
        .astype(np.float32, copy=True)
    )
    b_std = float(
        linear.bias[0].detach().cpu().item()
    )

    # Fold scaler into a raw-feature linear layer.
    w_raw = (w_std / std).astype(np.float32)
    b_raw = float(
        b_std
        - np.dot(w_raw, mean)
    )

    return {
        "w_raw": w_raw,
        "b_raw": b_raw,
        "mean": mean,
        "std": std,
        "n_pos": int(len(X_pos)),
        "n_neg": int(len(X_neg)),
        "n_hard": int(n_hard_target),
        "n_random": int(n_random_target),
    }


,name,hard_fraction,lambda_rank
0,hard25_lambda0p0,0.25,0.0
1,hard25_lambda0p1,0.25,0.1
2,hard25_lambda0p3,0.25,0.3
3,hard25_lambda1p0,0.25,1.0
4,hard50_lambda0p0,0.50,0.0
5,hard50_lambda0p1,0.50,0.1
6,hard50_lambda0p3,0.50,0.3
7,hard50_lambda1p0,0.50,1.0


In [9]:
SUBMISSION_LEVELS = 100_001


def probability_bin_5dp(probabilities):
    return np.rint(
        np.clip(probabilities, 0.0, 1.0) * 100_000
    ).astype(np.int32)


def update_histogram(hist, indices):
    values, counts = np.unique(indices, return_counts=True)
    hist[values] += counts.astype(np.int64)


def ap_from_histograms(positive_hist, total_hist):
    positives = int(positive_hist.sum())

    if positives <= 0:
        return float("nan")

    pos_desc = positive_hist[::-1]
    total_desc = total_hist[::-1]

    tp = np.cumsum(pos_desc, dtype=np.int64)
    predicted_positive = np.cumsum(total_desc, dtype=np.int64)

    precision = tp / np.maximum(predicted_positive, 1)

    return float(
        np.sum(
            (pos_desc / positives) * precision
        )
    )


def dense_rank_pearson(prediction, target):
    prediction = np.asarray(prediction)
    target = np.asarray(target)

    if len(prediction) < 2:
        return float("nan")

    return float(
        pearsonr(
            rankdata(prediction, method="dense"),
            rankdata(target, method="dense"),
        )[0]
    )


def score_linear_raw(features, w_raw, b_raw):
    logit = features @ w_raw + b_raw
    return sigmoid_numpy(logit)


## 3. Cross-contig CV внутри train

На каждой fold:

```text
8 train-contigs → head training
1 train-contig  → exhaustive natural-background AP
```

Все 8 candidate heads оцениваются на одном проходе backbone.

Primary selection metric: **mean AP**.


In [10]:
cv_heads = {}

for fold_i, heldout_contig in enumerate(TRAIN_CONTIGS, 1):
    print()
    print("=" * 80)
    print(
        f"TRAIN HEADS FOLD {fold_i}/{len(TRAIN_CONTIGS)} "
        f"heldout={heldout_contig}"
    )
    print("=" * 80)

    fold_pos_X = pos_X[pos_contig != heldout_contig]
    fold_random_X = random_X[random_contig != heldout_contig]
    fold_hard_X = hard_X[hard_contig != heldout_contig]

    cv_heads[heldout_contig] = {}

    for config_i, config in enumerate(CONFIGS, 1):
        print(
            f"  {config_i}/{len(CONFIGS)} {config['name']}"
        )

        trained = train_linear_head(
            train_pos_X=fold_pos_X,
            train_random_X=fold_random_X,
            train_hard_X=fold_hard_X,
            hard_fraction=config["hard_fraction"],
            lambda_rank=config["lambda_rank"],
            seed=HEAD_SEED + 1000 * fold_i + config_i,
        )

        cv_heads[heldout_contig][config["name"]] = trained

print("All fold heads trained.")



TRAIN HEADS FOLD 1/9 heldout=NC_064035.1
  1/8 hard25_lambda0p0
  2/8 hard25_lambda0p1
  3/8 hard25_lambda0p3
  4/8 hard25_lambda1p0
  5/8 hard50_lambda0p0
  6/8 hard50_lambda0p1
  7/8 hard50_lambda0p3
  8/8 hard50_lambda1p0

TRAIN HEADS FOLD 2/9 heldout=NC_064036.1
  1/8 hard25_lambda0p0
  2/8 hard25_lambda0p1
  3/8 hard25_lambda0p3
  4/8 hard25_lambda1p0
  5/8 hard50_lambda0p0
  6/8 hard50_lambda0p1
  7/8 hard50_lambda0p3
  8/8 hard50_lambda1p0

TRAIN HEADS FOLD 3/9 heldout=NC_064038.1
  1/8 hard25_lambda0p0
  2/8 hard25_lambda0p1
  3/8 hard25_lambda0p3
  4/8 hard25_lambda1p0
  5/8 hard50_lambda0p0
  6/8 hard50_lambda0p1
  7/8 hard50_lambda0p3
  8/8 hard50_lambda1p0

TRAIN HEADS FOLD 4/9 heldout=NC_064039.1
  1/8 hard25_lambda0p0
  2/8 hard25_lambda0p1
  3/8 hard25_lambda0p3
  4/8 hard25_lambda1p0
  5/8 hard50_lambda0p0
  6/8 hard50_lambda0p1
  7/8 hard50_lambda0p3
  8/8 hard50_lambda1p0

TRAIN HEADS FOLD 5/9 heldout=NC_064040.1
  1/8 hard25_lambda0p0
  2/8 hard25_lambda0p1
  3/8 ha

In [11]:
CV_RESULTS_PATH = RUN_DIR / "cross_contig_cv.csv"

cv_rows = []

frozen_model.eval()

for fold_i, heldout_contig in enumerate(TRAIN_CONTIGS, 1):
    print()
    print("=" * 80)
    print(
        f"EVAL FOLD {fold_i}/{len(TRAIN_CONTIGS)} "
        f"{heldout_contig}"
    )
    print("=" * 80)

    fold_tiles = (
        train_tiles[
            train_tiles["contig"].astype(str) == heldout_contig
        ]
        .reset_index(drop=True)
    )

    states = {
        config["name"]: {
            "total_hist": np.zeros(SUBMISSION_LEVELS, dtype=np.int64),
            "positive_hist": np.zeros(SUBMISSION_LEVELS, dtype=np.int64),
            "positive_score": [],
            "positive_count": [],
        }
        for config in CONFIGS
    }

    started = time.perf_counter()
    positions_seen = 0

    with torch.no_grad():
        for row_i in range(len(fold_tiles)):
            selected = fold_tiles.iloc[row_i:row_i + 1]
            batch = train_generator.make_batch(selected)

            features, _, _ = extract_backbone_and_old_outputs(
                frozen_model,
                batch,
            )

            plus_count = (
                batch.count[0, 0, :]
                .numpy().astype(np.float32, copy=False)
            )
            minus_count_reverse = (
                batch.count[0, 1, :]
                .flip(-1)
                .numpy().astype(np.float32, copy=False)
            )

            plus_mask = (
                batch.mask[0, 0, :]
                .numpy().astype(bool, copy=False)
            )
            minus_mask_reverse = (
                batch.mask[0, 1, :]
                .flip(-1)
                .numpy().astype(bool, copy=False)
            )

            for orient_i, (count, mask) in enumerate(
                (
                    (plus_count, plus_mask),
                    (minus_count_reverse, minus_mask_reverse),
                )
            ):
                valid_index = np.flatnonzero(mask)
                valid_feature = features[orient_i, valid_index]
                valid_count = count[valid_index]
                positive = valid_count > 0

                for config in CONFIGS:
                    name = config["name"]
                    head = cv_heads[heldout_contig][name]

                    probability = score_linear_raw(
                        valid_feature,
                        head["w_raw"],
                        head["b_raw"],
                    )

                    score_bin = probability_bin_5dp(probability)
                    state = states[name]

                    update_histogram(
                        state["total_hist"],
                        score_bin,
                    )

                    if positive.any():
                        update_histogram(
                            state["positive_hist"],
                            score_bin[positive],
                        )

                        state["positive_score"].append(
                            probability[positive].copy()
                        )
                        state["positive_count"].append(
                            valid_count[positive].copy()
                        )

                positions_seen += int(mask.sum())

            del (
                batch,
                selected,
                features,
                plus_count,
                minus_count_reverse,
                plus_mask,
                minus_mask_reverse,
            )

            if (
                row_i == 0
                or (row_i + 1) % 250 == 0
                or row_i + 1 == len(fold_tiles)
            ):
                elapsed = time.perf_counter() - started
                print(
                    f"tiles={row_i + 1:>5}/{len(fold_tiles)} "
                    f"positions={positions_seen:>12,} "
                    f"positions/s={positions_seen / elapsed:,.0f}"
                )

            if (row_i + 1) % 50 == 0:
                gc.collect()

    for config in CONFIGS:
        name = config["name"]
        state = states[name]

        positive_score = np.concatenate(state["positive_score"])
        positive_count = np.concatenate(state["positive_count"])

        cv_rows.append(
            {
                "heldout_contig": heldout_contig,
                "config": name,
                "hard_fraction": config["hard_fraction"],
                "lambda_rank": config["lambda_rank"],
                "average_precision_5dp": ap_from_histograms(
                    state["positive_hist"],
                    state["total_hist"],
                ),
                "positive_spearman_raw": dense_rank_pearson(
                    positive_score,
                    positive_count,
                ),
                "positive_n": int(len(positive_count)),
            }
        )

cv_results = pd.DataFrame(cv_rows)
cv_results.to_csv(CV_RESULTS_PATH, index=False)

display(cv_results)
print("Saved:", CV_RESULTS_PATH)



EVAL FOLD 1/9 NC_064035.1
tiles=    1/18918 positions=         498 positions/s=23,827
tiles=  250/18918 positions=     374,670 positions/s=170,348
tiles=  500/18918 positions=     811,248 positions/s=183,401
tiles=  750/18918 positions=   1,209,988 positions/s=182,576
tiles= 1000/18918 positions=   1,606,742 positions/s=181,249
tiles= 1250/18918 positions=   2,025,652 positions/s=179,053
tiles= 1500/18918 positions=   2,447,804 positions/s=179,082
tiles= 1750/18918 positions=   2,875,410 positions/s=181,301
tiles= 2000/18918 positions=   3,276,578 positions/s=181,835
tiles= 2250/18918 positions=   3,727,040 positions/s=184,285
tiles= 2500/18918 positions=   4,184,270 positions/s=186,300
tiles= 2750/18918 positions=   4,626,284 positions/s=186,984
tiles= 3000/18918 positions=   5,062,232 positions/s=187,243
tiles= 3250/18918 positions=   5,524,502 positions/s=188,354
tiles= 3500/18918 positions=   5,960,950 positions/s=188,929
tiles= 3750/18918 positions=   6,421,794 positions/s=190,17

,heldout_contig,config,hard_fraction,lambda_rank,average_precision_5dp,positive_spearman_raw,positive_n
0,NC_064035.1,hard25_lambda0p0,0.25,0.0,0.011681,0.116434,30987
1,NC_064035.1,hard25_lambda0p1,0.25,0.1,0.007600,0.085000,30987
2,NC_064035.1,hard25_lambda0p3,0.25,0.3,0.004517,0.044979,30987
3,NC_064035.1,hard25_lambda1p0,0.25,1.0,0.001686,-0.050061,30987
4,NC_064035.1,hard50_lambda0p0,0.50,0.0,0.003047,0.007560,30987
...,...,...,...,...,...,...,...
67,NC_064048.1,hard25_lambda1p0,0.25,1.0,0.001102,-0.044106,10888
68,NC_064048.1,hard50_lambda0p0,0.50,0.0,0.001990,0.005764,10888
69,NC_064048.1,hard50_lambda0p1,0.50,0.1,0.001615,-0.009867,10888
70,NC_064048.1,hard50_lambda0p3,0.50,0.3,0.001114,-0.036378,10888


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER\run_001\cross_contig_cv.csv


In [12]:
cv_summary = (
    cv_results
    .groupby(
        ["config", "hard_fraction", "lambda_rank"],
        as_index=False,
    )
    .agg(
        mean_AP=("average_precision_5dp", "mean"),
        min_AP=("average_precision_5dp", "min"),
        max_AP=("average_precision_5dp", "max"),
        mean_positive_spearman=("positive_spearman_raw", "mean"),
    )
    .sort_values(
        ["mean_AP", "min_AP"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(cv_summary)

BEST_CONFIG_NAME = str(cv_summary.iloc[0]["config"])

BEST_CONFIG = next(
    config
    for config in CONFIGS
    if config["name"] == BEST_CONFIG_NAME
)

CV_SUMMARY_PATH = RUN_DIR / "cross_contig_cv_summary.csv"
cv_summary.to_csv(CV_SUMMARY_PATH, index=False)

print("SELECTED:", BEST_CONFIG)
print("Saved:", CV_SUMMARY_PATH)


,config,hard_fraction,lambda_rank,mean_AP,min_AP,max_AP,mean_positive_spearman
0,hard25_lambda0p0,0.25,0.0,0.009860,0.007784,0.011681,0.110448
1,hard25_lambda0p1,0.25,0.1,0.006264,0.004914,0.007600,0.080463
2,hard25_lambda0p3,0.25,0.3,0.003524,0.002686,0.004517,0.035786
3,hard50_lambda0p0,0.50,0.0,0.002464,0.001913,0.003047,0.006119
4,hard50_lambda0p1,0.50,0.1,0.002012,0.001428,0.002433,-0.011117
5,hard50_lambda0p3,0.50,0.3,0.001444,0.001099,0.001786,-0.044654
6,hard25_lambda1p0,0.25,1.0,0.001332,0.001031,0.001686,-0.054872
7,hard50_lambda1p0,0.50,1.0,0.000823,0.000618,0.001033,-0.098212


SELECTED: {'name': 'hard25_lambda0p0', 'hard_fraction': 0.25, 'lambda_rank': 0.0}
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER\run_001\cross_contig_cv_summary.csv


## 4. Final head на всех 9 train-contigs


In [13]:
FINAL_HEAD = train_linear_head(
    train_pos_X=pos_X,
    train_random_X=random_X,
    train_hard_X=hard_X,
    hard_fraction=BEST_CONFIG["hard_fraction"],
    lambda_rank=BEST_CONFIG["lambda_rank"],
    seed=999_037,
)

FINAL_HEAD_PATH = RUN_DIR / "final_linear_presence_head.npz"

np.savez(
    FINAL_HEAD_PATH,
    w_raw=FINAL_HEAD["w_raw"],
    b_raw=np.array([FINAL_HEAD["b_raw"]], dtype=np.float32),
    hard_fraction=np.array(
        [BEST_CONFIG["hard_fraction"]],
        dtype=np.float32,
    ),
    lambda_rank=np.array(
        [BEST_CONFIG["lambda_rank"]],
        dtype=np.float32,
    ),
)

print("Selected:", BEST_CONFIG)
print("Final head:", FINAL_HEAD_PATH)
print(
    {
        key: FINAL_HEAD[key]
        for key in ("n_pos", "n_neg", "n_hard", "n_random")
    }
)


Selected: {'name': 'hard25_lambda0p0', 'hard_fraction': 0.25, 'lambda_rank': 0.0}
Final head: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER\run_001\final_linear_presence_head.npz
{'n_pos': 170443, 'n_neg': 681772, 'n_hard': 170443, 'n_random': 511329}


## 5. Один полный validation

Считаем:

- baseline WIDTH64 AP;
- reranker AP;
- AP по каждому validation-contig;
- baseline presence Spearman;
- reranker Spearman на positives;
- frozen intensity Spearman;
- top 1/5/10%;
- baseline HARD_FP cohort retention;
- LOW_TP recall как guardrail.


In [14]:
BUDGETS = (0.01, 0.05, 0.10)

VALIDATION_HARD_CANDIDATES_PER_TILE_STRAND = 4
VALIDATION_HARD_FP_KEEP = 20_000


def new_eval_state():
    return {
        "baseline_total_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "baseline_positive_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "rerank_total_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "rerank_positive_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "positive_baseline_score": [],
        "positive_rerank_score": [],
        "positive_intensity": [],
        "positive_count": [],
        "hard_candidate_baseline_score": [],
        "hard_candidate_rerank_score": [],
    }


def threshold_for_top_fraction(total_hist, fraction):
    total = int(total_hist.sum())
    target = int(math.ceil(fraction * total))

    desc = total_hist[::-1]
    cumulative = np.cumsum(desc, dtype=np.int64)

    idx = int(
        np.searchsorted(
            cumulative,
            target,
            side="left",
        )
    )

    bin_index = SUBMISSION_LEVELS - 1 - idx
    selected = int(cumulative[idx])

    return {
        "bin_index": int(bin_index),
        "threshold": float(bin_index / 100_000.0),
        "selected": selected,
        "selected_fraction": float(selected / total),
    }


def budget_stats(total_hist, positive_hist, fraction):
    threshold = threshold_for_top_fraction(
        total_hist,
        fraction,
    )

    b = threshold["bin_index"]

    selected = int(total_hist[b:].sum())
    tp = int(positive_hist[b:].sum())
    fp = selected - tp

    positives = int(positive_hist.sum())

    return {
        **threshold,
        "tp": tp,
        "fp": fp,
        "precision": float(tp / max(selected, 1)),
        "recall": float(tp / max(positives, 1)),
    }


In [15]:
segment_names = ["overall", *VALIDATION_CONTIGS]

eval_states = {
    segment: new_eval_state()
    for segment in segment_names
}

positions_seen = 0
started = time.perf_counter()

frozen_model.eval()

with torch.no_grad():
    for row_i in range(len(validation_tiles)):
        selected = validation_tiles.iloc[row_i:row_i + 1]
        contig = str(selected.iloc[0]["contig"])

        batch = validation_generator.make_batch(selected)

        features, old_logit, old_intensity = (
            extract_backbone_and_old_outputs(
                frozen_model,
                batch,
            )
        )

        baseline_probability = sigmoid_numpy(old_logit)

        rerank_probability = np.stack(
            [
                score_linear_raw(
                    features[orient_i],
                    FINAL_HEAD["w_raw"],
                    FINAL_HEAD["b_raw"],
                )
                for orient_i in range(features.shape[0])
            ],
            axis=0,
        )

        plus_count = (
            batch.count[0, 0, :]
            .numpy().astype(np.float32, copy=False)
        )
        minus_count_reverse = (
            batch.count[0, 1, :]
            .flip(-1)
            .numpy().astype(np.float32, copy=False)
        )

        plus_mask = (
            batch.mask[0, 0, :]
            .numpy().astype(bool, copy=False)
        )
        minus_mask_reverse = (
            batch.mask[0, 1, :]
            .flip(-1)
            .numpy().astype(bool, copy=False)
        )

        for orient_i, (count, mask) in enumerate(
            (
                (plus_count, plus_mask),
                (minus_count_reverse, minus_mask_reverse),
            )
        ):
            valid_index = np.flatnonzero(mask)

            valid_count = count[valid_index]
            positive = valid_count > 0

            baseline_score = baseline_probability[
                orient_i,
                valid_index,
            ]
            rerank_score = rerank_probability[
                orient_i,
                valid_index,
            ]
            intensity_score = old_intensity[
                orient_i,
                valid_index,
            ]

            baseline_bin = probability_bin_5dp(baseline_score)
            rerank_bin = probability_bin_5dp(rerank_score)

            for segment in ("overall", contig):
                state = eval_states[segment]

                update_histogram(
                    state["baseline_total_hist"],
                    baseline_bin,
                )
                update_histogram(
                    state["rerank_total_hist"],
                    rerank_bin,
                )

                if positive.any():
                    update_histogram(
                        state["baseline_positive_hist"],
                        baseline_bin[positive],
                    )
                    update_histogram(
                        state["rerank_positive_hist"],
                        rerank_bin[positive],
                    )

                    state["positive_baseline_score"].append(
                        baseline_score[positive].copy()
                    )
                    state["positive_rerank_score"].append(
                        rerank_score[positive].copy()
                    )
                    state["positive_intensity"].append(
                        intensity_score[positive].copy()
                    )
                    state["positive_count"].append(
                        valid_count[positive].copy()
                    )

            negative_index = np.flatnonzero(~positive)

            if len(negative_index):
                n_hard = min(
                    VALIDATION_HARD_CANDIDATES_PER_TILE_STRAND,
                    len(negative_index),
                )

                negative_baseline = baseline_score[negative_index]

                if len(negative_index) <= n_hard:
                    hard_local = negative_index
                else:
                    top = np.argpartition(
                        negative_baseline,
                        -n_hard,
                    )[-n_hard:]
                    hard_local = negative_index[top]

                for segment in ("overall", contig):
                    eval_states[segment][
                        "hard_candidate_baseline_score"
                    ].append(
                        baseline_score[hard_local].copy()
                    )

                    eval_states[segment][
                        "hard_candidate_rerank_score"
                    ].append(
                        rerank_score[hard_local].copy()
                    )

            positions_seen += int(mask.sum())

        del (
            batch,
            selected,
            features,
            old_logit,
            old_intensity,
            baseline_probability,
            rerank_probability,
            plus_count,
            minus_count_reverse,
            plus_mask,
            minus_mask_reverse,
        )

        if (
            row_i == 0
            or (row_i + 1) % 100 == 0
            or row_i + 1 == len(validation_tiles)
        ):
            elapsed = time.perf_counter() - started
            print(
                f"tiles={row_i + 1:>6}/{len(validation_tiles)} "
                f"positions={positions_seen:>13,} "
                f"positions/s={positions_seen / elapsed:,.0f}"
            )

        if (row_i + 1) % 25 == 0:
            gc.collect()

print("FULL VALIDATION FINISHED")


tiles=     1/47743 positions=          210 positions/s=26,910
tiles=   100/47743 positions=      125,828 positions/s=154,112
tiles=   200/47743 positions=      269,398 positions/s=154,921
tiles=   300/47743 positions=      403,094 positions/s=153,443
tiles=   400/47743 positions=      552,594 positions/s=156,183
tiles=   500/47743 positions=      708,204 positions/s=159,064
tiles=   600/47743 positions=      871,458 positions/s=162,565
tiles=   700/47743 positions=    1,020,090 positions/s=162,692
tiles=   800/47743 positions=    1,185,336 positions/s=165,083
tiles=   900/47743 positions=    1,346,300 positions/s=166,522
tiles=  1000/47743 positions=    1,519,462 positions/s=168,995
tiles=  1100/47743 positions=    1,700,978 positions/s=171,877
tiles=  1200/47743 positions=    1,853,300 positions/s=171,609
tiles=  1300/47743 positions=    2,017,896 positions/s=172,381
tiles=  1400/47743 positions=    2,193,246 positions/s=172,601
tiles=  1500/47743 positions=    2,381,142 positions/s=1

In [16]:
metric_rows = []
positive_cache = {}
hard_cache = {}

for segment, state in eval_states.items():
    positive_baseline_score = np.concatenate(
        state["positive_baseline_score"]
    )
    positive_rerank_score = np.concatenate(
        state["positive_rerank_score"]
    )
    positive_intensity = np.concatenate(
        state["positive_intensity"]
    )
    positive_count = np.concatenate(
        state["positive_count"]
    )

    hard_baseline = np.concatenate(
        state["hard_candidate_baseline_score"]
    )
    hard_rerank = np.concatenate(
        state["hard_candidate_rerank_score"]
    )

    keep = min(
        VALIDATION_HARD_FP_KEEP,
        len(hard_baseline),
    )

    if len(hard_baseline) > keep:
        idx = np.argpartition(
            hard_baseline,
            -keep,
        )[-keep:]

        hard_baseline = hard_baseline[idx]
        hard_rerank = hard_rerank[idx]

    baseline_ap = ap_from_histograms(
        state["baseline_positive_hist"],
        state["baseline_total_hist"],
    )
    reranker_ap = ap_from_histograms(
        state["rerank_positive_hist"],
        state["rerank_total_hist"],
    )

    metric_rows.append(
        {
            "segment": segment,
            "positive_n": int(len(positive_count)),
            "baseline_AP_5dp": baseline_ap,
            "reranker_AP_5dp": reranker_ap,
            "delta_AP": reranker_ap - baseline_ap,
            "baseline_presence_spearman_raw": dense_rank_pearson(
                positive_baseline_score,
                positive_count,
            ),
            "reranker_spearman_raw": dense_rank_pearson(
                positive_rerank_score,
                positive_count,
            ),
            "frozen_intensity_spearman_raw": dense_rank_pearson(
                positive_intensity,
                positive_count,
            ),
        }
    )

    positive_cache[segment] = {
        "baseline_score": positive_baseline_score,
        "rerank_score": positive_rerank_score,
        "count": positive_count,
    }

    hard_cache[segment] = {
        "baseline_score": hard_baseline,
        "rerank_score": hard_rerank,
    }

final_metrics = (
    pd.DataFrame(metric_rows)
    .sort_values("segment")
    .reset_index(drop=True)
)

FINAL_METRICS_PATH = RUN_DIR / "final_validation_metrics.csv"
final_metrics.to_csv(FINAL_METRICS_PATH, index=False)

display(final_metrics)
print("Saved:", FINAL_METRICS_PATH)


,segment,positive_n,baseline_AP_5dp,reranker_AP_5dp,delta_AP,baseline_presence_spearman_raw,reranker_spearman_raw,frozen_intensity_spearman_raw
0,NC_064034.1,23529,0.033783,0.007611,-0.026171,0.191207,0.106980,0.203072
1,NC_064041.1,13984,0.028973,0.007121,-0.021852,0.221014,0.145913,0.234122
2,NC_064042.1,18692,0.038690,0.010077,-0.028614,0.206500,0.115241,0.220363
3,overall,56205,0.033997,0.008222,-0.025775,0.177041,0.102931,0.188709


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER\run_001\final_validation_metrics.csv


In [17]:
overall_state = eval_states["overall"]
overall_positive = positive_cache["overall"]
overall_hard = hard_cache["overall"]

baseline_positive_score = overall_positive["baseline_score"]
rerank_positive_score = overall_positive["rerank_score"]

low_tp_cutoff = float(
    np.quantile(
        baseline_positive_score,
        0.25,
    )
)
low_tp_mask = baseline_positive_score <= low_tp_cutoff

budget_rows = []

for fraction in BUDGETS:
    baseline_stats = budget_stats(
        overall_state["baseline_total_hist"],
        overall_state["baseline_positive_hist"],
        fraction,
    )
    rerank_stats = budget_stats(
        overall_state["rerank_total_hist"],
        overall_state["rerank_positive_hist"],
        fraction,
    )

    baseline_threshold = baseline_stats["threshold"]
    rerank_threshold = rerank_stats["threshold"]

    baseline_low_tp_recall = float(
        np.mean(
            baseline_positive_score[low_tp_mask]
            >= baseline_threshold
        )
    )
    rerank_low_tp_recall = float(
        np.mean(
            rerank_positive_score[low_tp_mask]
            >= rerank_threshold
        )
    )

    hard_remain_baseline = int(
        np.sum(
            overall_hard["baseline_score"]
            >= baseline_threshold
        )
    )
    hard_remain_rerank = int(
        np.sum(
            overall_hard["rerank_score"]
            >= rerank_threshold
        )
    )

    budget_rows.append(
        {
            "top_fraction": fraction,

            "baseline_actual_selected_fraction": (
                baseline_stats["selected_fraction"]
            ),
            "baseline_precision": baseline_stats["precision"],
            "baseline_recall": baseline_stats["recall"],
            "baseline_FP": baseline_stats["fp"],

            "reranker_actual_selected_fraction": (
                rerank_stats["selected_fraction"]
            ),
            "reranker_precision": rerank_stats["precision"],
            "reranker_recall": rerank_stats["recall"],
            "reranker_FP": rerank_stats["fp"],

            "baseline_hardFP_cohort_remaining": hard_remain_baseline,
            "reranker_hardFP_cohort_remaining": hard_remain_rerank,

            "baseline_LOW_TP_recall": baseline_low_tp_recall,
            "reranker_LOW_TP_recall": rerank_low_tp_recall,
            "delta_LOW_TP_recall": (
                rerank_low_tp_recall
                - baseline_low_tp_recall
            ),
        }
    )

budget_diagnostics = pd.DataFrame(budget_rows)

BUDGET_PATH = RUN_DIR / "top_budget_diagnostics.csv"
budget_diagnostics.to_csv(BUDGET_PATH, index=False)

display(budget_diagnostics)

print("Baseline LOW_TP cutoff:", low_tp_cutoff)
print("LOW_TP n:", int(low_tp_mask.sum()))
print("Saved:", BUDGET_PATH)


,top_fraction,baseline_actual_selected_fraction,baseline_precision,baseline_recall,baseline_FP,reranker_actual_selected_fraction,reranker_precision,reranker_recall,reranker_FP,baseline_hardFP_cohort_remaining,reranker_hardFP_cohort_remaining,baseline_LOW_TP_recall,reranker_LOW_TP_recall,delta_LOW_TP_recall
0,0.01,0.010000,0.022838,0.340539,818936,0.010001,0.012769,0.190410,827446,20000,9079,0.0,0.005124,0.005124
1,0.05,0.050002,0.008167,0.608914,4156320,0.050001,0.006105,0.455173,4164859,20000,16359,0.0,0.066539,0.066539
2,0.10,0.100002,0.004963,0.740041,8339283,0.100002,0.004173,0.622204,8345943,20000,18764,0.0,0.174281,0.174281


Baseline LOW_TP cutoff: 0.5802421569824219
LOW_TP n: 14052
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER\run_001\top_budget_diagnostics.csv


## 6. Result JSON

Главный вопрос:

> превратился ли найденный 64-D TP/HARD_FP signal в реальный AP gain на естественном фоне?


In [18]:
WIDTH64_REFERENCE_AP = 0.03399689188641901

overall_row = (
    final_metrics[
        final_metrics["segment"] == "overall"
    ]
    .iloc[0]
)

contig_rows = final_metrics[
    final_metrics["segment"] != "overall"
]

improved_contigs = int(
    np.sum(contig_rows["delta_AP"] > 0)
)

summary = {
    "experiment": {
        "id": EXPERIMENT_ID,
        "architecture": (
            "frozen WIDTH64 RF1029 backbone + "
            "linear 64->1 presence reranker"
        ),
        "selected_config": BEST_CONFIG,
        "cv_selection_metric": (
            "mean natural-background AP on held-out train contigs"
        ),
        "low_tp_optimized": False,
    },
    "validation": {
        "reference_width64_AP": WIDTH64_REFERENCE_AP,
        "baseline_recomputed_AP": float(
            overall_row["baseline_AP_5dp"]
        ),
        "reranker_AP": float(
            overall_row["reranker_AP_5dp"]
        ),
        "delta_AP": float(
            overall_row["delta_AP"]
        ),
        "improved_validation_contigs": improved_contigs,
        "validation_contigs_total": int(len(contig_rows)),
        "baseline_presence_spearman_raw": float(
            overall_row["baseline_presence_spearman_raw"]
        ),
        "reranker_spearman_raw": float(
            overall_row["reranker_spearman_raw"]
        ),
        "frozen_intensity_spearman_raw": float(
            overall_row["frozen_intensity_spearman_raw"]
        ),
    },
    "files": {
        "train_feature_bank": str(FEATURE_BANK_PATH),
        "cross_contig_cv": str(CV_RESULTS_PATH),
        "cross_contig_cv_summary": str(CV_SUMMARY_PATH),
        "final_head": str(FINAL_HEAD_PATH),
        "final_validation_metrics": str(FINAL_METRICS_PATH),
        "top_budget_diagnostics": str(BUDGET_PATH),
    },
}

RESULT_JSON = RUN_DIR / "EXP037_RESULT.json"

RESULT_JSON.write_text(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print("=" * 80)
print("EXP037 COMPLETE")
print("=" * 80)
print()
print(
    json.dumps(
        summary["validation"],
        ensure_ascii=False,
        indent=2,
    )
)
print()
print("Send me:")
print(RESULT_JSON)
print(BUDGET_PATH)
print(CV_SUMMARY_PATH)


EXP037 COMPLETE

{
  "reference_width64_AP": 0.03399689188641901,
  "baseline_recomputed_AP": 0.03399689188641901,
  "reranker_AP": 0.008222069686748124,
  "delta_AP": -0.025774822199670887,
  "improved_validation_contigs": 0,
  "validation_contigs_total": 3,
  "baseline_presence_spearman_raw": 0.17704136669635773,
  "reranker_spearman_raw": 0.10293099284172058,
  "frozen_intensity_spearman_raw": 0.18870866298675537
}

Send me:
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER\run_001\EXP037_RESULT.json
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER\run_001\top_budget_diagnostics.csv
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER\run_001\cross_contig_cv_summary.csv


## 7. Optional registration

Compatibility-layer со старым experiment reporter уже встроен.


In [19]:
SAVE_EXPERIMENT = False

REPORT_METRICS_PATH = (
    RUN_DIR
    / "validation_metrics_report.csv"
)

report_metrics = (
    final_metrics[
        [
            "segment",
            "reranker_AP_5dp",
            "reranker_spearman_raw",
        ]
    ]
    .rename(
        columns={
            "reranker_AP_5dp": "average_precision",
            "reranker_spearman_raw": "epic_spearman",
        }
    )
)

report_metrics.to_csv(
    REPORT_METRICS_PATH,
    index=False,
)

CARD_PATH = (
    PROJECT_ROOT
    / "cnn_exp"
    / f"{EXPERIMENT_ID}.md"
)

CARD_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if not CARD_PATH.exists():
    CARD_PATH.write_text(
        f'''---
id: {EXPERIMENT_ID}
type: experiment
experiment_type: cnn-candidate
status: completed
decision: review
date: 2026-09-24
seed: 42
split_version: contig_holdout_v1
primary_metric: average_precision
secondary_metric: epic_spearman
reference_experiment: CNN-EXP-036
tags:
  - ml/experiment
  - ml/cnn
  - ml/hard-negative
  - ml/ranking
---

# {EXPERIMENT_ID} — Frozen WIDTH64 Hard-FP Reranker

← [[cnn_exp/_index.md|Реестр CNN]]

## Pre-registration

- **Гипотеза:** hard-negative sampling + ranking objective улучшат
  natural-background AP при frozen WIDTH64 backbone.
- **Единственное изменение:** новая linear presence-head и objective/sampling.
- **Что заморожено:** WIDTH64 backbone, RF1029, split, sequence representation,
  old intensity-head.
- **Reference:** [[cnn_exp/CNN-EXP-036.md|CNN-EXP-036]].

## Автоматический отчёт

<!-- auto:cnn-experiment-report:start -->

<!-- auto:cnn-experiment-report:end -->

## Ручные заметки

-
''',
        encoding="utf-8",
    )

print("Reporter metrics:", REPORT_METRICS_PATH)
print("Card:", CARD_PATH)


Reporter metrics: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER\run_001\validation_metrics_report.csv
Card: D:\Projects\EPIC\EPIC\cnn_exp\CNN-EXP-037.md


In [20]:
if SAVE_EXPERIMENT:
    from ml_project.cnn_experiment_report import (
        finalize_cnn_experiment,
    )

    reference_report_metrics = (
        EXP036_RUN
        / "validation_metrics_report.csv"
    )

    assert reference_report_metrics.is_file(), (
        "Нет EXP036 validation_metrics_report.csv. "
        "Сначала выполни compatibility adapter в EXP036."
    )

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id=EXPERIMENT_ID,
        title="Frozen WIDTH64 Hard-FP Reranker",
        hypothesis=(
            "Hard-negative sampling plus pairwise ranking "
            "can improve natural-background AP using a "
            "frozen WIDTH64 representation."
        ),
        notebook_path=(
            "notebooks/experiments/CNN_experiments/"
            "CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER.ipynb"
        ),
        artifact_dir=RUN_DIR,
        training_history=(
            EXP036_RUN
            / "training_history.csv"
        ),
        candidate_metrics=REPORT_METRICS_PATH,
        reference_experiment="CNN-EXP-036",
        reference_metrics=reference_report_metrics,
        decision="review",
        notes=(
            "Frozen WIDTH64 backbone. Linear presence reranker. "
            "LOW_TP diagnostic only."
        ),
    )

    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)

else:
    print(
        "SAVE_EXPERIMENT=False. "
        "Сначала пришли EXP037_RESULT.json, "
        "top_budget_diagnostics.csv и "
        "cross_contig_cv_summary.csv."
    )


SAVE_EXPERIMENT=False. Сначала пришли EXP037_RESULT.json, top_budget_diagnostics.csv и cross_contig_cv_summary.csv.
